# Занятие 5: первое обучение
Новых данных нет. Используйте свои features.csv, labels.csv и split.csv четвёртого этапа. Списки колонок в config_example.json — пример по преподавательскому набору: замените их собственными именами признаков и именем долга для baseline. Не создавайте фиктивные колонки ради запуска.
Тест в этом ноутбуке не оценивается. Пример обучает одну конфигурацию; полный журнал двух методов и ограниченных настроек готовит команда.


In [ ]:
from pathlib import Path
import json
import numpy as np
from training_utils import load_train_validation, campaign_metrics
from model_factory import build_pipeline

PREPARED = Path("../../lesson4_prepared")  # Измените на свои результаты этапа 4.
CONFIG = Path("../config_example.json")    # Запуск из lesson5/starter.
cfg = json.loads(CONFIG.read_text(encoding="utf-8"))
parts = load_train_validation(PREPARED, cfg["numeric_columns"], cfg["categorical_columns"])
for name, part in parts.items():
    print(name, part["X"].shape)


In [ ]:
# Один первый опыт с LogisticRegression из подготовленного плана.
experiment = cfg["experiments"][0]
model = build_pipeline(experiment, cfg["numeric_columns"], cfg["categorical_columns"], cfg["seed"])
model.fit(parts["train"]["X"], parts["train"]["y"])
positive = int(np.flatnonzero(model.classes_ == 1)[0])
validation_score = model.predict_proba(parts["validation"]["X"])[:, positive]
summary, by_date, ranked = campaign_metrics(parts["validation"]["keys"], parts["validation"]["y"], validation_score, cfg["campaign_fraction"])
print(summary)
print(by_date)


## Работа команды
- Воспроизведите baseline на тех же train и validation и всей базе снимка.
- Обучите два выбранных на этапе 4 метода. Выполните 3–6 осмысленных конфигураций суммарно (максимум 8).
- Для каждого опыта пересоздайте Pipeline, сохраните параметры, train/validation-метрики, время и риск переобучения.
- Заранее задайте правило выбора на validation. Сохраните выбранный Pipeline целиком, конфигурацию, список X и версии.
- В новом процессе восстановите validation-прогнозы через check_saved_model.py; адаптируйте структуру bundle по примеру ниже.
- Тест и переобучение на train+validation оставьте до следующего этапа и согласования протокола.


In [ ]:
# Пример структуры сохранения своего выбранного опыта после сравнения:
# import joblib
# bundle = {"pipeline": selected_pipeline,
#           "numeric_columns": cfg["numeric_columns"],
#           "categorical_columns": cfg["categorical_columns"],
#           "selected_experiment": selected_id, "config": cfg}
# joblib.dump(bundle, output_path)
# Нельзя подставлять текущий model как победителя без выполнения сравнения.
